# RAG Hôpital — Doublons, similarité Jaccard et provenance des documents

Objectif : comparer l'exercice `duplicate(4).ipynb` et `07_indexation_avancee.ipynb` aux **chunks réels** de RAG Hôpital, détecter des doublons, calculer Jaccard et vérifier les références aux sources.

1. Exécuter depuis le terminal de la racine du projet : `python scripts/audit_duplicates.py --emit-safe-copy`.
2. **Valider** : `python scripts/validate_dedup_preview.py`.
3. Exécuter les cellules ci-dessous avec le kernel `.venv` de RAG Hôpital.

**Aucun changement dans Qdrant.** Les paires similaires détectées par MinHashLSH ne sont pas supprimées automatiquement.

In [1]:
from pathlib import Path
import csv
import json

ROOT = Path.cwd().resolve()
if not (ROOT / "data/chunks/chunks.jsonl").exists():
    ROOT = ROOT.parent

assert (ROOT / "data/chunks/chunks.jsonl").exists(), "Exécuter depuis le projet rag-hopital ou son dossier notebooks/"
REPORT = ROOT / "reports/dedup_audit"
assert (REPORT / "SYNTHESE.md").exists(), "Lancer d'abord : python scripts/audit_duplicates.py"
print("Projet :", ROOT)
print("Rapports :", REPORT)


Projet : /home/bm/formation/RAG/rag-hopital
Rapports : /home/bm/formation/RAG/rag-hopital/reports/dedup_audit


In [2]:
from IPython.display import display, Markdown

display(Markdown((REPORT / "SYNTHESE.md").read_text(encoding="utf-8")))


# RAG Hôpital — Audit des doublons (AVANT Qdrant)

## Méthode
- Chunks existants : `data/chunks/chunks.jsonl` (lecture seule).
- Fichiers source : `data/hopital` (lecture seule).
- Documents exacts : SHA-256 sur les octets DOCX/PDF originaux.
- Chunks exacts : SHA-256 sur le texte brut, avec ligne patient.
- Chunks identiques APRÈS normalisation : SHA-256 du texte normalisé : audit seulement, 
  aucune fusion fondée sur ce hash (la ponctuation et les décimales peuvent être significatives).
- Doublons approximatifs : shingles de 3 mots normalisés, 
  MinHash/LSH (seuil de candidats 0.65) puis Jaccard EXACT des candidats.
- Pour la similarité seulement, la ligne patient ajoutée au chunk est ignorée.
- LSH est probabiliste : des paires similaires peuvent ne pas être proposées.
- Aucun chunk et aucun document n'est supprimé des sources, aucun index modifié.

## Résultats
- Fichiers DOCX/PDF examinés : **280**.
- Groupes de fichiers identiques (SHA-256) : **0**.
- Documents reconstruits des chunks : **280**.
- Chunks examinés : **1173**.
- Groupes de chunks strictement identiques : **0**.
- Groupes de chunks identiques après normalisation : **0**.
- Dont groupes avec textes bruts DISTINCTS (variantes de format ou autre) : **0**.
- Chunks à normalisation vide exclus de ce hash : 0.
- Variantes normalisées à examiner : `reports/dedup_audit/chunks_normalized_format_variants.csv`.
- Documents écartés de l'audit LSH (trop courts) : 0.
- Chunks écartés de l'audit LSH (trop courts) : 72.
- Écran LSH : activé, probabiliste.
- Paires de documents candidates LSH : **184**.
- Paires de chunks candidates LSH : **2263**.
- Paires de documents Jaccard >= 0.80 parmi les candidates : **0**.
- Paires de chunks Jaccard >= 0.80 parmi les candidates : **1691**.
- Paires de chunks Jaccard >= 0.90 parmi les candidates : **1082**.
- Paires de chunks cross-patient >= 0.80 parmi les candidates : **1127**.
- Portée de la consolidation expérimentale : **same-document**.
- Chunks dans la copie : **1173**.
- Occurrences fusionnées dans la copie uniquement : **0**.
- Groupes fusionnés : **0**, dont **0** multi-documents.
- Chaque chunk conservé porte `source_references` (toutes les occurrences, avec `document`, `chunk_id`, `date`, `patient_ipp`, `title`, etc.), `source_documents`, `source_chunk_ids` et `duplicate_count`.
- Copie distincte : `data/chunks/chunks_dedup_preview.jsonl`.
- Audit des regroupements : `reports/dedup_audit/preview_merged_groups.csv`.
- Contrôle obligatoire : `python scripts/validate_dedup_preview.py`.

## Vérifications supplémentaires du cours 07

- Hash normalisé : détecte des variantes textuelles ; aucune suppression automatique.
- Le formateur illustre les bigrammes (`--k 2`) et un seuil LSH de 0.2 sur quatre petits textes ; les réglages par défaut de cet audit de corpus sont des trigrammes et un seuil de 0.65 pour limiter le volume de candidats.
- MMR agit à la recherche (dense + sparse → RRF → MMR), et non sur le corpus avant Qdrant. Prévoir une expérience séparée après cet audit.
- Les résultats Jaccard affichés sont EXACTS pour les paires proposées par LSH, mais LSH ne garantit pas de retrouver toutes les paires.


## Documents les plus similaires

| ID A | ID B | Jaccard | Patients différents ? |
|---|---|---:|---|
| 4100006_2025-10-28_sortie.docx | 4100036_2025-08-20_sortie.docx | 0.775 | Oui |
| 4100004_2025-12-07_sortie.docx | 4100044_2025-06-16_sortie.docx | 0.771 | Oui |
| 4100024_2025-04-29_sortie.docx | 4100034_2025-06-03_sortie.docx | 0.763 | Oui |
| 4100026_2026-04-06_sortie.docx | 4100046_2026-05-22_sortie.docx | 0.757 | Oui |
| 4100009_2025-10-12_sortie.docx | 4100049_2025-04-22_sortie.docx | 0.750 | Oui |
| 4100006_2025-10-28_sortie.docx | 4100056_2025-04-02_sortie.docx | 0.749 | Oui |
| 4100005_2025-06-14_sortie.docx | 4100045_2025-03-18_sortie.docx | 0.748 | Oui |
| 4100004_2025-12-07_sortie.docx | 4100014_2025-02-24_sortie.docx | 0.747 | Oui |
| 4100014_2025-02-24_sortie.docx | 4100044_2025-06-16_sortie.docx | 0.747 | Oui |
| 4100005_2025-06-14_sortie.docx | 4100025_2025-07-18_sortie.docx | 0.740 | Oui |
| 4100025_2025-07-18_sortie.docx | 4100045_2025-03-18_sortie.docx | 0.740 | Oui |
| 4100009_2025-10-12_sortie.docx | 4100039_2025-07-16_sortie.docx | 0.739 | Oui |

## Chunks les plus similaires

| ID A | ID B | Jaccard | Patients différents ? |
|---|---|---:|---|
| 4100001_2025-06-30_bilan.docx::2 | 4100011_2025-02-23_bilan.docx::2 | 1.000 | Oui |
| 4100001_2025-06-30_bilan.docx::2 | 4100021_2025-04-11_bilan.docx::2 | 1.000 | Oui |
| 4100001_2025-06-30_bilan.docx::2 | 4100031_2025-09-01_bilan.docx::2 | 1.000 | Oui |
| 4100001_2025-06-30_bilan.docx::2 | 4100041_2025-01-08_bilan.docx::2 | 1.000 | Oui |
| 4100001_2025-06-30_bilan.docx::2 | 4100051_2025-12-23_bilan.docx::2 | 1.000 | Oui |
| 4100001_2025-07-07_consultation.docx::2 | 4100021_2025-04-18_consultation.docx::2 | 1.000 | Oui |
| 4100001_2025-07-07_consultation.docx::3 | 4100002_2025-04-13_consultation.docx::3 | 1.000 | Oui |
| 4100001_2025-07-07_consultation.docx::3 | 4100006_2025-07-03_consultation.docx::2 | 1.000 | Oui |
| 4100001_2025-07-07_consultation.docx::3 | 4100009_2025-08-20_consultation.docx::2 | 1.000 | Oui |
| 4100001_2025-07-07_consultation.docx::3 | 4100010_2025-10-31_consultation.docx::2 | 1.000 | Oui |
| 4100001_2025-07-07_consultation.docx::3 | 4100011_2025-03-02_consultation.docx::3 | 1.000 | Oui |
| 4100001_2025-07-07_consultation.docx::3 | 4100012_2025-11-24_consultation.docx::3 | 1.000 | Oui |

## Décision avant déduplication

1. Examiner les paires à haut Jaccard : identité de patient, dates, médicaments, chiffres, négations.
2. Jamais de fusion inter-patients ; fusion inter-documents uniquement en mode same-patient explicite, avec IPP connu ET texte brut identique.
3. Chaque chunk consolidé conserve toutes les références sources et dates; adapter l'évaluateur au champ source_references avant une comparaison A/B.
4. Si une déduplication est retenue : créer une NOUVELLE collection Qdrant, pas modifier la baseline.
5. Comparer la nouvelle indexation aux métriques E11 (Hit@5, Fact Recall@5, nDCG@5).


In [3]:
with (REPORT / "chunks_similar_lsh_candidates.csv").open(encoding="utf-8", newline="") as f:
    candidates = list(csv.DictReader(f))

print("Paires candidates :", len(candidates))
for p in candidates[:15]:
    print(f"{float(p['jaccard']):.3f} | {p['id_a']}  ↔  {p['id_b']}"
          f" | Patients distincts : {p['cross_patient']}")


Paires candidates : 2263
1.000 | 4100001_2025-06-30_bilan.docx::2  ↔  4100011_2025-02-23_bilan.docx::2 | Patients distincts : True
1.000 | 4100001_2025-06-30_bilan.docx::2  ↔  4100021_2025-04-11_bilan.docx::2 | Patients distincts : True
1.000 | 4100001_2025-06-30_bilan.docx::2  ↔  4100031_2025-09-01_bilan.docx::2 | Patients distincts : True
1.000 | 4100001_2025-06-30_bilan.docx::2  ↔  4100041_2025-01-08_bilan.docx::2 | Patients distincts : True
1.000 | 4100001_2025-06-30_bilan.docx::2  ↔  4100051_2025-12-23_bilan.docx::2 | Patients distincts : True
1.000 | 4100001_2025-07-07_consultation.docx::2  ↔  4100021_2025-04-18_consultation.docx::2 | Patients distincts : True
1.000 | 4100001_2025-07-07_consultation.docx::3  ↔  4100002_2025-04-13_consultation.docx::3 | Patients distincts : True
1.000 | 4100001_2025-07-07_consultation.docx::3  ↔  4100006_2025-07-03_consultation.docx::2 | Patients distincts : True
1.000 | 4100001_2025-07-07_consultation.docx::3  ↔  4100009_2025-08-20_consultation.d

## Inspecter une paire réelle

Les paires candidates sont classées par Jaccard décroissant. La cellule suivante affiche les deux textes et leurs identifiants de source. Vérifie **les chiffres, les noms, les dates, les médicaments et les négations** avant de proposer une exclusion.


In [4]:
chunk_by_id = {}
with (ROOT / "data/chunks/chunks.jsonl").open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            obj = json.loads(line)
            chunk_by_id[obj["chunk_id"]] = obj

if candidates:
    pair = candidates[0]  # Changer l'indice pour inspecter une autre paire.
    print("Jaccard :", pair["jaccard"])
    for side in ("a", "b"):
        chunk = chunk_by_id[pair[f"id_{side}"]]
        print("\n===", pair[f"id_{side}"], "===")
        print("Patient IPP:", chunk.get("patient_ipp"))
        print("Document:", chunk["document"])
        print(chunk["text"][:2500])
else:
    print("Aucune paire candidate LSH : examine aussi les CSV de doublons SHA-256.")


Jaccard : 1.0

=== 4100001_2025-06-30_bilan.docx::2 ===
Patient IPP: 4100001
Document: 4100001_2025-06-30_bilan.docx
Patient: Mme Anne DUBOIS | IPP: 4100001
Bilan biologique
Commentaire du biologiste
Hyperglycémie chronique. Fonction rénale conservée.

=== 4100011_2025-02-23_bilan.docx::2 ===
Patient IPP: 4100011
Document: 4100011_2025-02-23_bilan.docx
Patient: M. Jacques CLEMENT | IPP: 4100011
Bilan biologique
Commentaire du biologiste
Hyperglycémie chronique. Fonction rénale conservée.


## Interpréter, puis comparer au baseline E11

- **Identique (SHA-256) :** examen des métadonnées et de la provenance avant toute exclusion.
- **Jaccard ≥ 0,80 :** candidat à examiner, **pas** décision automatique.
- **Patients différents :** conserver, sauf démonstration documentaire très spécifique.
- **Même patient, dates différentes :** conserver si les données évoluent dans le temps.
- **Suppression expérimentale :** produire une copie séparée et une **nouvelle collection** Qdrant, puis comparer Hit@5, Fact Recall@5 et nDCG@5 aux résultats E11.


## Métadonnées et références conservées

Même si deux chunks ont exactement le même texte, chaque occurrence documentaire doit être traçable : document, date, titre, patient, ancien `chunk_id`. Le nouveau fichier `chunks_dedup_preview.jsonl` ajoute `source_references` à chaque chunk conservé. Les documents appartenant à des patients différents restent distincts.

In [5]:
preview_path = ROOT / "data/chunks/chunks_dedup_preview.jsonl"
assert preview_path.exists(), "Lancer d'abord python scripts/audit_duplicates.py --emit-safe-copy"

preview = [json.loads(line) for line in preview_path.open(encoding="utf-8") if line.strip()]
merged = [record for record in preview if record["duplicate_count"] > 0]
print("Chunks originaux :", len(chunk_by_id))
print("Chunks conservés dans la COPIE :", len(preview))
print("Occurrences fusionnées :", sum(record["duplicate_count"] for record in preview))
print("Groupes consolidés :", len(merged))
print("Groupes multi-documents :", sum(r["source_document_count"] > 1 for r in merged))

Chunks originaux : 1173
Chunks conservés dans la COPIE : 1173
Occurrences fusionnées : 0
Groupes consolidés : 0
Groupes multi-documents : 0


In [6]:
if merged:
    kept = merged[0]
    print("Chunk conservé :", kept["chunk_id"])
    print("Documents d'origine :", kept["source_documents"])
    print("Nombre d'occurrences :", kept["occurrence_count"])
    print("Références complètes :")
    for ref in kept["source_references"]:
        print("  -", ref.get("document"), "| chunk:", ref.get("chunk_id"),
              "| patient:", ref.get("patient_ipp"), "| date:", ref.get("date"),
              "| titre:", ref.get("title"))
else:
    print("Aucun doublon exact fusionné par la règle prudente : normal si le corpus n'en contient pas.")

Aucun doublon exact fusionné par la règle prudente : normal si le corpus n'en contient pas.


## Comparer un couple similaire avec le notebook du formateur

La valeur `jaccard` ci-dessous est calculée **exactement sur les shingles** (trigrammes de mots normalisés) des deux textes. MinHashLSH a seulement servi à retrouver des paires candidates ; il ne donne pas directement ce score exact. Un score élevé ne justifie pas une suppression médicale.

In [7]:
import sys
sys.path.insert(0, str(ROOT / "scripts"))
from audit_duplicates import shingles, jaccard, without_patient_header

if candidates:
    pair = candidates[0]
    a = chunk_by_id[pair["id_a"]]
    b = chunk_by_id[pair["id_b"]]
    sha = shingles(without_patient_header(a["text"]), 3)
    shb = shingles(without_patient_header(b["text"]), 3)
    print("Intersection:", len(sha & shb))
    print("Union:", len(sha | shb))
    print("Jaccard recalculé:", round(jaccard(sha, shb), 6))
    print("Jaccard rapport:", pair["jaccard"])
    print("Mêmes patients ?", a.get("patient_ipp") == b.get("patient_ipp"))
else:
    print("Pas de paire candidate LSH. Revoir le rapport exact SHA-256.")

Intersection: 8
Union: 8
Jaccard recalculé: 1.0
Jaccard rapport: 1.0
Mêmes patients ? False


## Décision à prendre avant toute nouvelle indexation

- Lire `reports/dedup_audit/preview_merged_groups.csv`.
- Vérifier les dates, doses, négations et identifiants patients des groupes significatifs.
- Confirmer la validation de provenance (`PASS`).
- **Ne pas** remplacer `chunks.jsonl` ni lancer le script d'indexation existant sur cette copie.
- Si le gain paraît plausible, créer plus tard une nouvelle collection Qdrant, adapter l'évaluateur aux références multiples et comparer les mesures E11 sans toucher à la baseline.

Mode multi-documents **seulement après revue** : `python scripts/audit_duplicates.py --emit-safe-copy --preview-scope same-patient`, puis validation avec le même `--preview-scope`. Les fichiers de résultats d'expérience sont régénérés à chaque lancement.

## Audit des hash normalisés (cours 07)

Le formateur détecte les variantes en normalisant le texte **avant** comparaison. Le fichier `chunks_normalized_format_variants.csv` contient les groupes où les textes sont différents à l'octet près, mais identiques après normalisation. C'est un **signal de revue**, jamais une règle de fusion médicale. Le hash brut reste la condition de la consolidation expérimentale.

In [8]:
with (REPORT / "chunks_normalized_format_variants.csv").open(encoding="utf-8", newline="") as f:
    normalized_variants = list(csv.DictReader(f))
print("Groupes identiques après normalisation, mais différents en texte brut :", len(normalized_variants))
for row in normalized_variants[:10]:
    print(row["copies"], "occurrences |", row["ids"][:170])

Groupes identiques après normalisation, mais différents en texte brut : 0


## Aligner le test avec les paramètres du formateur

Dans `07_indexation_avancee.ipynb`, les démonstrations utilisent des **bigrammes** (`n=2`), **128 permutations MinHash** et, dans l'exemple à quatre textes, un **seuil LSH de 0,2**. L'audit Hôpital adopte par défaut `--k 3` et `--lsh-threshold 0.65` pour limiter les faux candidats à grande échelle. Le seuil LSH est un *filtre probabiliste*, pas une règle de suppression.

Pour reproduire les réglages de la démonstration sur les chunks réels :

```bash
python scripts/audit_duplicates.py --k 2 --lsh-threshold 0.2 --emit-safe-copy
python scripts/validate_dedup_preview.py
```

Attention : cela **régénère les rapports du même répertoire** et peut créer de nombreuses paires à inspecter. Pour conserver deux expériences, utiliser l'option `--reports` avec un dossier différent ; le fichier de preview reste partagé en cas de `--emit-safe-copy`. Lancer de préférence la première analyse avec les paramètres par défaut.

## MMR : **phase distincte, non exécutée ici**

La dernière section du notebook du formateur présente **Maximal Marginal Relevance (MMR)**, au moment de la recherche. Pour notre pipeline hybride, l'ordre de traitement à tester serait :

**Dense + BM25 → RRF → MMR → résultats top-k**

On ne recopie pas directement les connexions et noms de collection du professeur (`Veltis`, port `6333`, serveur d'embeddings externe), car **RAG Hôpital** utilise une autre collection, un port Qdrant différent et des mesures E11 déjà établies. Ne pas introduire MMR et déduplication au cours de la même expérience : on ne saurait pas à quoi attribuer une amélioration des scores.

Les deux thèmes présentent une proximité technique mais pas le même objectif : **déduplication avant indexation ; MMR lors de la récupération**.

In [10]:
import json
from pathlib import Path
from difflib import unified_diff

path = ROOT / "data/chunks/chunks.jsonl"

chunks = {}

with path.open(encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        chunks[record["chunk_id"]] = record

pairs = [
    (
        "4100001_2025-06-30_bilan.docx::2",
        "4100011_2025-02-23_bilan.docx::2"
    ),
    (
        "4100001_2025-07-07_consultation.docx::0",
        "4100001_2025-07-07_ordonnance.docx::0"
    )
]

for a, b in pairs:
    print("\n" + "=" * 70)
    print("COMPARAISON")
    print(a)
    print(b)

    text_a = chunks[a]["text"]
    text_b = chunks[b]["text"]

    print("\nDIFFÉRENCES ENTRE LES TEXTES :")

    differences = unified_diff(
        text_a.splitlines(),
        text_b.splitlines(),
        fromfile=a,
        tofile=b,
        lineterm=""
    )

    print("\n".join(differences))


COMPARAISON
4100001_2025-06-30_bilan.docx::2
4100011_2025-02-23_bilan.docx::2

DIFFÉRENCES ENTRE LES TEXTES :
--- 4100001_2025-06-30_bilan.docx::2
+++ 4100011_2025-02-23_bilan.docx::2
@@ -1,4 +1,4 @@
-Patient: Mme Anne DUBOIS | IPP: 4100001
+Patient: M. Jacques CLEMENT | IPP: 4100011
 Bilan biologique
 Commentaire du biologiste
 Hyperglycémie chronique. Fonction rénale conservée.

COMPARAISON
4100001_2025-07-07_consultation.docx::0
4100001_2025-07-07_ordonnance.docx::0

DIFFÉRENCES ENTRE LES TEXTES :
--- 4100001_2025-07-07_consultation.docx::0
+++ 4100001_2025-07-07_ordonnance.docx::0
@@ -1,5 +1,5 @@
 Patient: Mme Anne DUBOIS | IPP: 4100001
-Compte rendu de consultation
+Ordonnance
 Centre Hospitalier de Valmont — Service de Endocrinologie
 Patient : Mme Anne DUBOIS, née le 16/10/1962 (62 ans) — IPP 4100001
 Date : 07/07/2025


In [11]:
import json

path = ROOT / "data/chunks/chunks.jsonl"

chunks = [
    json.loads(line)
    for line in path.open(encoding="utf-8")
]

# Trier les chunks selon leur longueur
chunks_sorted = sorted(
    chunks,
    key=lambda c: len(c["text"].split())
)

print("Total chunks :", len(chunks))

print(
    "Chunks de 30 mots ou moins :",
    sum(
        len(c["text"].split()) <= 30
        for c in chunks
    )
)

print("\nLes 10 chunks les plus courts :")

for chunk in chunks_sorted[:10]:
    print(
        chunk["chunk_id"],
        "|",
        len(chunk["text"].split()),
        "mots"
    )

Total chunks : 1173
Chunks de 30 mots ou moins : 628

Les 10 chunks les plus courts :
notice_bisoprolol.docx::1 | 8 mots
notice_salbutamol.docx::4 | 8 mots
notice_pristinamycine.docx::6 | 8 mots
notice_paracetamol.docx::4 | 8 mots
notice_metronidazole.docx::4 | 8 mots
notice_sitagliptine.docx::4 | 9 mots
notice_amlodipine.docx::1 | 9 mots
notice_amlodipine.docx::2 | 9 mots
notice_furosemide.docx::1 | 9 mots
notice_paracetamol.docx::1 | 9 mots


In [12]:
from collections import Counter

# Répartition des chunks courts par type de document
types = Counter()

for chunk in chunks:
    text = chunk["text"]
    words = len(text.split())

    if words <= 30:
        document = chunk["document"]

        if document.startswith("notice_"):
            category = "notice"
        else:
            category = document.rsplit("_", 1)[-1].split(".")[0]

        types[category] += 1

print("CHUNKS COURTS PAR TYPE DE DOCUMENT")

for category, count in types.most_common():
    print(f"{category}: {count}")

print("\n" + "=" * 60)
print("CONTENU DES 10 CHUNKS LES PLUS COURTS")

for chunk in chunks_sorted[:10]:
    print("\nDOCUMENT :", chunk["chunk_id"])
    print("MOTS :", len(chunk["text"].split()))
    print("TEXTE :")
    print(chunk["text"])
    print("-" * 60)

CHUNKS COURTS PAR TYPE DE DOCUMENT
consultation: 229
notice: 204
sortie: 86
imagerie: 48
bilan: 24
cro: 24
coronarographie: 12
ordonnance: 1

CONTENU DES 10 CHUNKS LES PLUS COURTS

DOCUMENT : notice_bisoprolol.docx::1
MOTS : 8
TEXTE :
Fiche médicament : Bisoprolol
Classe thérapeutique
Bêtabloquant cardiosélectif.
------------------------------------------------------------

DOCUMENT : notice_salbutamol.docx::4
MOTS : 8
TEXTE :
Fiche médicament : Salbutamol
Contre-indications
Hypersensibilité au salbutamol.
------------------------------------------------------------

DOCUMENT : notice_pristinamycine.docx::6
MOTS : 8
TEXTE :
Fiche médicament : Pristinamycine
Interactions médicamenteuses
Ciclosporine, colchicine.
------------------------------------------------------------

DOCUMENT : notice_paracetamol.docx::4
MOTS : 8
TEXTE :
Fiche médicament : Paracétamol
Contre-indications
Insuffisance hépatocellulaire sévère.
------------------------------------------------------------

DOCUMENT : n

In [13]:
consultations = [
    c for c in chunks_sorted
    if c["document"].endswith("_consultation.docx")
    and len(c["text"].split()) <= 30
]

print("Consultations courtes :", len(consultations))

for chunk in consultations[:8]:
    print("\n" + "=" * 60)
    print("DOCUMENT :", chunk["chunk_id"])
    print("MOTS :", len(chunk["text"].split()))
    print("TEXTE :")
    print(chunk["text"])

Consultations courtes : 229

DOCUMENT : 4100001_2025-07-07_consultation.docx::3
MOTS : 16
TEXTE :
Patient: Mme Anne DUBOIS | IPP: 4100001
Compte rendu de consultation
Allergies
Pas d'allergie médicamenteuse connue.

DOCUMENT : 4100002_2025-04-13_consultation.docx::3
MOTS : 16
TEXTE :
Patient: M. Bruno MARTIN | IPP: 4100002
Compte rendu de consultation
Allergies
Pas d'allergie médicamenteuse connue.

DOCUMENT : 4100006_2025-07-03_consultation.docx::2
MOTS : 16
TEXTE :
Patient: M. Marcel THOMAS | IPP: 4100006
Compte rendu de consultation
Allergies
Pas d'allergie médicamenteuse connue.

DOCUMENT : 4100009_2025-08-20_consultation.docx::2
MOTS : 16
TEXTE :
Patient: M. André GUERIN | IPP: 4100009
Compte rendu de consultation
Allergies
Pas d'allergie médicamenteuse connue.

DOCUMENT : 4100010_2025-10-31_consultation.docx::2
MOTS : 16
TEXTE :
Patient: M. Luc ROBERT | IPP: 4100010
Compte rendu de consultation
Allergies
Pas d'allergie médicamenteuse connue.

DOCUMENT : 4100011_2025-03-02_consult

In [14]:
allergy_chunks = [
    c for c in consultations
    if "Allergies" in c["text"]
]

print("Consultations courtes :", len(consultations))
print("Chunks consacrés aux allergies :", len(allergy_chunks))

print("\nInformations les plus fréquentes :")

from collections import Counter

contents = Counter()

for c in allergy_chunks:
    lines = c["text"].splitlines()
    if lines:
        contents[lines[-1]] += 1

for text, count in contents.most_common(10):
    print(count, ":", text)

Consultations courtes : 229
Chunks consacrés aux allergies : 48

Informations les plus fréquentes :
36 : Pas d'allergie médicamenteuse connue.
3 : Allergie à la pénicilline (urticaire généralisée en 2009).
3 : Allergie alimentaire à l'arachide (œdème des lèvres en 2018).
3 : Allergie aux produits de contraste iodés (réaction cutanée lors d'un scanner).
2 : Intolérance à la codéine (nausées et vomissements importants).
1 : Allergie à l'aspirine (bronchospasme).
